# Imports 

In [5]:
import pandas as pd

from fragility.config import DEMO_DIR
from fragility.universe import load_universe, load_holdings
from fragility.portfolio import load_portfolio
from fragility.lookthrough import (
    company_exposure, sector_exposure, country_exposure, overlap_matrix,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## Load the reference data 

In [6]:
universe = load_universe()
holdings = load_holdings()

print(f"{len(universe)} funds in the universe, "
      f"{holdings['etf_ticker'].nunique()} with holdings data")

20 funds in the universe, 20 with holdings data


## Load the overlapping demo portfolio 

In [7]:
result = load_portfolio(DEMO_DIR / "overlapping.csv", universe)
portfolio = result.portfolio

print("Errors: ", result.errors or "none")
print("Notices:", result.notices or "none")
portfolio

c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#d

Errors:  none
Notices: none


,ticker,name,type,value_gbp,weight,holding_id
0,VWRL.L,Vanguard FTSE All-World,etf,6000.0,0.24,VWRL.L
1,VUSA.L,Vanguard S&P 500 (Dist),etf,5000.0,0.20,VUSA.L
2,EQQQ.L,Invesco EQQQ Nasdaq-100,etf,5000.0,0.20,EQQQ.L
3,CSPX.L,iShares Core S&P 500,etf,4000.0,0.16,CSPX.L
4,IITU.L,iShares S&P 500 IT Sector,etf,2000.0,0.08,IITU.L
5,AAPL,Apple Inc.,share,1500.0,0.06,APPLE
6,NVDA,NVIDIA Corporation,share,1500.0,0.06,NVIDIA


## the top 10 companies (the actual checklist item)

In [8]:
exposure = company_exposure(portfolio, holdings, universe)
companies = exposure[exposure["kind"] == "company"]   # leave out the "Other" bucket
top10 = companies.head(10)

show = top10[["company", "exposure", "exposure_gbp", "via"]].copy()
show["exposure"] = (show["exposure"] * 100).round(1).astype(str) + "%"
show["exposure_gbp"] = show["exposure_gbp"].map("£{:,.0f}".format)
show.index = range(1, len(show) + 1)
show

,company,exposure,exposure_gbp,via
1,NVIDIA Corp,13.4%,"£3,338","CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L, direct"
2,Apple Inc,12.9%,"£3,237","CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L, direct"
3,Microsoft Corp,5.4%,"£1,358","CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
4,Alphabet Inc,3.7%,£913,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
5,Amazon.com Inc,2.7%,£671,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
6,Micron Technology Inc,2.1%,£533,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
7,Advanced Micro Devices Inc,1.8%,£460,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
8,Broadcom Inc,1.8%,£446,"CSPX.L, IITU.L, VUSA.L, VWRL.L"
9,Meta Platforms Inc,1.8%,£439,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
10,Tesla Inc,1.5%,£369,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"


## Check 

In [9]:
residual = exposure.loc[exposure["kind"] == "residual", "exposure"].sum()
blocks = exposure[exposure["kind"] == "block"]
multi = companies[companies["n_routes"] >= 2]

print(f"Total exposure:          {exposure['exposure'].sum():.4f}   (must be 1.0000)")
print(f"Top 10 companies:        {top10['exposure'].sum():.1%} of your money "
      f"(£{top10['exposure_gbp'].sum():,.0f} of £{portfolio['value_gbp'].sum():,.0f})")
print(f"Funds' smaller holdings: {residual:.1%}")
print(f"Reached via 2+ routes:   {len(multi)} companies")
print("Funds not looked through:", blocks["company"].tolist() or "none")

Total exposure:          1.0000   (must be 1.0000)
Top 10 companies:        47.1% of your money (£11,764 of £25,000)
Funds' smaller holdings: 29.2%
Reached via 2+ routes:   59 companies
Funds not looked through: none


## Sectors, countries and fund overlap 

In [10]:
display(sector_exposure(portfolio, holdings, universe).head(5))
display(country_exposure(portfolio, holdings, universe).head(5))

funds = portfolio.loc[portfolio["type"] == "etf", "ticker"].tolist()
overlap_matrix(funds, holdings).round(2)

,sector,exposure,exposure_gbp,via
0,Information Technology,0.463442,11586.055,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L, direct"
1,Unclassified (smaller holdings),0.292109,7302.721,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
2,Communication Services,0.059268,1481.695,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
3,Consumer Discretionary,0.044448,1111.196,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
4,Unknown,0.040768,1019.195,EQQQ.L


,country,exposure,exposure_gbp,via
0,United States,0.641643,16041.075,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
1,Unknown,0.221375,5534.376,"VWRL.L, direct"
2,Mixed (smaller holdings),0.136295,3407.376,VWRL.L
3,Switzerland,0.000687,17.178,VWRL.L


,VWRL.L,VUSA.L,EQQQ.L,CSPX.L,IITU.L
VWRL.L,1.00,0.37,0.26,0.36,0.19
VUSA.L,0.37,1.00,0.43,0.59,0.33
EQQQ.L,0.26,0.43,1.00,0.41,0.41
CSPX.L,0.36,0.59,0.41,1.00,0.32
IITU.L,0.19,0.33,0.41,0.32,1.00
